In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
stock = yf.download(
    "RELIANCE.NS",
    period="5y",
    interval="1d",
    auto_adjust=False
)
print(stock)
if isinstance(stock.columns, pd.MultiIndex):
    stock.columns = stock.columns.get_level_values(0)
print(stock.head())
stock["Return"] = stock["Close"].pct_change()
stock["MA5"] = stock["Close"].rolling(5).mean()
stock["MA20"] = stock["Close"].rolling(20).mean()
stock["Volume_Change"] = stock["Volume"].pct_change()
stock["Target"]=stock["Close"].shift(-1)
features = [
    "Close",
    "Volume",
    "Return",
    "MA5",
    "MA20",
    "Volume_Change"
]
X = stock[features]
y = stock["Target"]
X = X.replace([np.inf, -np.inf], np.nan)
data = pd.concat([X, y], axis=1)
data = data.dropna()
X = data[features]
y = data["Target"]
split = int(len(X) * 0.8)
X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]
model = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    objective="reg:squarederror",
    random_state=42
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)
mae = mean_absolute_error(y_test, prediction)

rmse = np.sqrt(mean_squared_error(y_test, prediction))

print("MAE:", mae)

print("RMSE:", rmse)
# 14. Predict next closing price

latest_data = X.iloc[[-1]]

next_price = model.predict(latest_data)[0]

print("Predicted next closing price:", next_price)

[*********************100%***********************]  1 of 1 completed


Price         Adj Close        Close         High          Low         Open  \
Ticker      RELIANCE.NS  RELIANCE.NS  RELIANCE.NS  RELIANCE.NS  RELIANCE.NS   
Date                                                                          
2021-09-28  1154.057495  1175.924683  1183.700928  1159.656738  1169.440552   
2021-09-29  1144.908691  1166.602417  1186.054565  1163.002686  1186.054565   
2021-09-30  1141.013306  1162.633423  1169.717529  1153.911133  1165.287109   
2021-10-01  1143.029053  1164.687134  1172.117310  1151.442139  1154.649536   
2021-10-04  1157.726196  1179.662842  1188.292847  1170.848145  1178.209106   
...                 ...          ...          ...          ...          ...   
2026-09-22  1240.400024  1240.400024  1251.900024  1237.400024  1247.599976   
2026-09-23  1248.000000  1248.000000  1252.800049  1238.900024  1242.000000   
2026-09-24  1219.199951  1219.199951  1241.599976  1219.199951  1236.599976   
2026-09-25  1226.000000  1226.000000  1227.400024  1